In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from groq import Groq

# Initialize the Groq client
client = Groq()

In [4]:
def llm(prompt):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
    messages=[
        {"role": "user", "content": prompt}
    ] )
    return response.choices[0].message.content

In [5]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

I'm excited that you're interested in the course. However, I need a bit more information from you. Could you please provide more context or details about the course you're referring to, such as what it's about, where it's being offered, and what the enrollment process is like? That way, I can give you a more accurate answer about whether you can join now.


In [6]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [7]:
prompt = f"""
Your task is to answer questions from the course participants based on the provided context.

Use the context to find relevant information to answer the question. If the context does not contain enough information to answer the question, respond with "I don't know".

Question:
{question}

Context:
{context}
"""

In [8]:
print(prompt)


Your task is to answer questions from the course participants based on the provided context.

Use the context to find relevant information to answer the question. If the context does not contain enough information to answer the question, respond with "I don't know".

Question:
I just discovered the course. Can I join now?

Context:

I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presen

In [9]:
question = "I just discovered the course. Can I join now?"
answer = llm(prompt)
print(answer)

You can join now. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [10]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [11]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [12]:
courses_raw

[{'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 404},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 41},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 85},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 255},
 {'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 472}]

In [13]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1350

In [14]:
documents[10]

{'id': '316180784f',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'}

In [15]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [16]:
search_results = index.search(question, filter_dict={"course": "llm-zoomcamp"}, num_results=5)

In [17]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [18]:
search_results = search(question)

In [19]:
print(search_results)

[{'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}, {'id': '977bf7786c', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?', 'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."}, {'id': '69d122f12e', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?', 'answer': 'No, you can only get a certificate 

In [20]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [21]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [22]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [23]:
context = build_context(search_results)
print(context)

General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.

You can only peer-review projects at the time the course is run

In [24]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question, 
        context=context
    )
    return prompt.strip()


In [25]:
prompt = build_prompt(question, search_results)
print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

In [26]:
response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
    messages=[
        {"role": "user", "content": prompt}
    ] )

In [27]:
response.choices[0].message.content

'Yes, you can join the course now. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted. You can start the course at any time and follow the weekly workflow by watching the lesson videos, working through the lesson notebooks/code, reading the homework instructions, and submitting your answers through the course platform before the deadline.'

In [28]:
response.usage

CompletionUsage(completion_tokens=76, prompt_tokens=510, total_tokens=586, completion_time=0.175494741, completion_tokens_details=None, prompt_time=0.027285059, prompt_tokens_details=None, queue_time=0.0508719, total_time=0.2027798)

In [40]:
message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
    messages=message_history)

In [41]:
response.choices[0].message.content

'Yes, you can join the course now. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted. You can start the course at any time by following the weekly workflow, which includes watching lesson videos, working through notebooks/code, and submitting homework before the deadline. Keep in mind that to get a certificate, you need to finish the course with a "live" cohort, which means you won\'t be able to get a certificate if you\'re taking the course in self-paced mode.'

In [42]:
def llm(instructions, user_prompt, model="llama-3.3-70b-versatile"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = client.chat.completions.create(
        model=model,
        messages=message_history
    )

    return response.choices[0].message.content

In [43]:
def rag(query, model="llama-3.3-70b-versatile"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [44]:
answer = rag(question)
print(answer)

You can join the course now, but if you want to receive a certificate, you'll need to submit your project while submissions are still being accepted. You can start learning and submitting homework immediately, without needing a confirmation email or registration check. Just follow the weekly workflow outlined in the course materials, starting with the LLM Zoomcamp docs, logistics docs, and GitHub repository.
